In [1]:
def main(datasources, start_date, end_date):
    """
    BigAlpha 2026：冲击后韧性因子（Shock-Resilience Alpha）

    仅使用当日及过去数据识别异常量价冲击；因子在收盘后形成，用于预测后续收益。
    返回列严格为：date, instrument, factor。
    """
    import numpy as np
    import pandas as pd
    import dai

    # 平台会在公榜/私榜自动替换物理表名，禁止硬编码行情表。
    bar1m = datasources["bar1m"]

    # 为“同一分钟、过去20个交易日”的基准留足历史。
    # 45个自然日通常覆盖约30个交易日。
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=45)

    sql = f"""
    WITH raw AS (
        SELECT
            date AS ts,
            date::DATE AS trade_date,
            instrument,
            strftime(date, '%H:%M:%S') AS bar_time,
            strftime(date, '%Y-%m-%d') AS trading_day,
            
            close,
            volume,
            ask_price1,
            bid_price1,
            COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0)
                + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0)
                + COALESCE(ask_volume5, 0) AS ask_depth,
            COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0)
                + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                + COALESCE(bid_volume5, 0) AS bid_depth
        FROM {bar1m}
        WHERE close > 0
    ),

    micro AS (
        SELECT
            *,
            CASE
                WHEN ask_price1 > 0 AND bid_price1 > 0 AND ask_price1 >= bid_price1
                THEN (ask_price1 - bid_price1)
                     / NULLIF((ask_price1 + bid_price1) / 2.0, 0)
                ELSE NULL
            END AS rel_spread,
            (bid_depth - ask_depth)
                / NULLIF(bid_depth + ask_depth, 0) AS book_imbalance,
            LAG(close) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS prev_close,
            LAG(volume) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS prev_cum_volume,
            ROW_NUMBER() OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS minute_index,
            COUNT(*) OVER (
                PARTITION BY instrument, trading_day
            ) AS minute_count
        FROM raw
    ),

    minute_bar AS (
        SELECT
            *,
            close / NULLIF(prev_close, 0) - 1.0 AS ret_1m,
            CASE
                WHEN prev_cum_volume IS NULL THEN GREATEST(volume, 0)
                WHEN volume >= prev_cum_volume THEN volume - prev_cum_volume
                ELSE GREATEST(volume, 0)
            END AS minute_volume
        FROM micro
    ),

    context AS (
        SELECT
            *,
            COUNT(LN(1.0 + minute_volume)) OVER (
                PARTITION BY instrument, bar_time
                ORDER BY trading_day
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS hist_count,
            AVG(LN(1.0 + minute_volume)) OVER (
                PARTITION BY instrument, bar_time
                ORDER BY trading_day
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS logvol_mean_20,
            STDDEV_SAMP(LN(1.0 + minute_volume)) OVER (
                PARTITION BY instrument, bar_time
                ORDER BY trading_day
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS logvol_std_20,
            AVG(ret_1m) OVER (
                PARTITION BY instrument, bar_time
                ORDER BY trading_day
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS ret_mean_20,
            STDDEV_SAMP(ret_1m) OVER (
                PARTITION BY instrument, bar_time
                ORDER BY trading_day
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) AS ret_std_20,
            LEAD(close, 15) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
            ) AS close_after_15,
            AVG(book_imbalance) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
                ROWS BETWEEN 1 FOLLOWING AND 5 FOLLOWING
            ) AS imbalance_after_5,
            AVG(rel_spread) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
                ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
            ) AS spread_before_5,
            AVG(rel_spread) OVER (
                PARTITION BY instrument, trading_day ORDER BY ts
                ROWS BETWEEN 5 FOLLOWING AND 15 FOLLOWING
            ) AS spread_after_5_15
        FROM minute_bar
    ),

    standardized AS (
        SELECT
            *,
            (LN(1.0 + minute_volume) - logvol_mean_20)
                / NULLIF(logvol_std_20, 0) AS volume_z,
            ABS(ret_1m - ret_mean_20)
                / NULLIF(ret_std_20, 0) AS return_z,
            CASE WHEN ret_1m > 0 THEN 1.0 ELSE -1.0 END AS shock_direction,
            LEAST(
                1.5,
                GREATEST(
                    0.0,
                    (CASE WHEN ret_1m > 0 THEN 1.0 ELSE -1.0 END)
                    * (close_after_15 / NULLIF(prev_close, 0) - 1.0)
                    / NULLIF(ABS(ret_1m), 0)
                )
            ) AS price_retention,
            LEAST(
                1.5,
                GREATEST(
                    0.5,
                    spread_before_5 / NULLIF(spread_after_5_15, 0)
                )
            ) AS liquidity_recovery
        FROM context
    ),

    scored AS (
        SELECT
            *,
            CASE
                WHEN hist_count >= 10
                 AND volume_z >= 1.5
                 AND return_z >= 1.5
                 AND ABS(ret_1m) <= 0.095
                 AND price_retention IS NOT NULL
                 AND liquidity_recovery IS NOT NULL
                 -- 防止上午事件的15个后续bar跨越午休。
                 AND (ts::TIME <= TIME '11:15:00' OR
                      (ts::TIME >= TIME '13:00:00'
                       AND ts::TIME <= TIME '14:45:00'))
                THEN
                    shock_direction
                    * SQRT(
                        LEAST(GREATEST(volume_z, 0.0), 5.0)
                        * LEAST(GREATEST(return_z, 0.0), 5.0)
                    )
                    * price_retention
                    * LEAST(
                        1.0,
                        GREATEST(
                            0.0,
                            0.5 + 0.5 * shock_direction
                                * COALESCE(imbalance_after_5, 0.0)
                        )
                    )
                    * liquidity_recovery
                    * (0.8 + 0.4 * minute_index / NULLIF(minute_count, 0))
                ELSE NULL
            END AS event_score
        FROM standardized
    )

    SELECT
        trade_date::DATETIME AS date,
        instrument,
        COALESCE(AVG(event_score), 0.0) AS factor
    FROM scored
    GROUP BY trade_date, instrument
    ORDER BY trade_date, instrument
    """

    factor = dai.query(
        sql,
        filters={"date": [query_start_date, end_date]},
        compression=True,
    ).df()

    # 精确裁剪评估区间；45日缓冲只用于计算历史基准。
    factor["date"] = pd.to_datetime(factor["date"])
    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    factor = factor[factor["date"].between(start_ts, end_ts)].copy()

    # 最后一道提交格式与数值安全检查。
    factor = factor[["date", "instrument", "factor"]]
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")
    factor["factor"] = factor["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    factor = (
        factor.drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    if list(factor.columns) != ["date", "instrument", "factor"]:
        raise ValueError("输出列必须且只能是 date、instrument、factor")
    if factor[["date", "instrument", "factor"]].isna().any().any():
        raise ValueError("因子输出中不应存在缺失值")

    # 与官方中证1000历史成分股表对齐，只保留当日真实成分股。
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    factor = pd.merge(
        factor,
        stk_pool[["date", "instrument"]],
        how="inner",
        on=["date", "instrument"],
    )
    factor = (
        factor[["date", "instrument", "factor"]]
        .drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    return factor
if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-07-20 23:09:50] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
